# E1 · W3 · Kaggle run 03a — bảng review nhãn v2 + kiểm tra lệch nhãn (CPU)
**Mục đích:** (1) tạo bảng review v2 cho 358 video trong `review_assignment_w2.csv` (issue #10); (2) đo độ nhạy lệch nhãn 1 frame và tạo bảng xem mù offset (issue #11). **Chưa xuất dữ liệu YOLO** (`--skip-export`).

**Trước khi chạy** (xem `docs/w3/kaggle_guide_w3.md`):
1. *Add Input*: dataset `e1-svanstrom-raw` và `e1-w3-tools` (bản mới nhất).
2. *Settings*: Accelerator **None**, Internet **On**.
3. Điền `COMMIT` ở ô dưới.

**Sau khi chạy:** *Save Version → Save & Run All (Commit)*. Tab *Output*: tải `w3_review_v2.zip` và `w3_export_evidence.zip`, giải nén vào `results/W03/kaggle_run_03a/`.

In [ ]:
COMMIT = ""   # <-- dan SHA day du 40 ky tu cua commit tren main

In [ ]:
import glob, hashlib, json, os, sys, zipfile
from pathlib import Path

def find(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, f"KHONG TIM THAY {name} trong /kaggle/input — kiem tra Add Input"
    if len(hits) > 1:
        print(f"[canh bao] {name} co {len(hits)} ban, dung ban dau: {hits}")
    return hits[0]

def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()

assert len(COMMIT) == 40 and all(c in "0123456789abcdef" for c in COMMIT.lower()), \
    "Dien COMMIT = SHA day du 40 ky tu cua commit tren main chua cac script (GitHub Desktop -> History -> Copy SHA)"

In [ ]:
!pip -q install mcos-decoder
import mcos_decoder; print("mcos-decoder OK")

In [ ]:
# Tim thu muc du lieu tho; neu dataset tho la file .zip thi giai nen vao /kaggle/working/raw
if not glob.glob("/kaggle/input/**/Data/Video_V", recursive=True) and not glob.glob("/kaggle/working/raw/**/Data/Video_V", recursive=True):
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        print("giai nen", z); zipfile.ZipFile(z).extractall("/kaggle/working/raw")
DATA_ROOT = "/kaggle/working/raw" if glob.glob("/kaggle/working/raw/**/Data/Video_V", recursive=True) else "/kaggle/input"
assert glob.glob(f"{DATA_ROOT}/**/Data/Video_V", recursive=True), "Khong thay Data/Video_V — kiem tra input e1-svanstrom-raw"
print("DATA_ROOT =", DATA_ROOT)

In [ ]:
TOOLS = {n: find(n) for n in ["w3_review_sheets.py", "w3_export_yolo.py", "video_split_v1.csv",
                             "annotation_audit.csv", "review_assignment_w2.csv"]}
for n, p in TOOLS.items():
    print(f"{n:28s} sha256={sha(p)[:16]}  {p}")

## 1. Bảng review nhãn v2 (358 video, vài phút)

In [ ]:
!{sys.executable} {TOOLS["w3_review_sheets.py"]} --data-root {DATA_ROOT} --review {TOOLS["review_assignment_w2.csv"]} --out /kaggle/working --commit {COMMIT}

## 2. Độ nhạy lệch nhãn + bảng xem mù offset (không xuất dữ liệu)

In [ ]:
!{sys.executable} {TOOLS["w3_export_yolo.py"]} --data-root {DATA_ROOT} --skip-export --split-csv {TOOLS["video_split_v1.csv"]} --audit {TOOLS["annotation_audit.csv"]} --review {TOOLS["review_assignment_w2.csv"]} --out /kaggle/working --commit {COMMIT}

## 3. Tự kiểm tra (phải in `PASS`)

In [ ]:
ok = True
rv = json.load(open("/kaggle/working/w3_review_v2/stage_status.json"))
print("review v2:", rv["sheets"]["videos_rendered"], "video /", rv["sheets"]["sheets"], "bang; loi:", len(rv["sheets"]["failed"]))
ok &= rv["sheets"]["videos_rendered"] == rv["sheets"]["videos_requested"] == 358 and not rv["sheets"]["failed"]
st = json.load(open("/kaggle/working/w3_export_evidence/stage_status.json"))
print({k: v["status"] for k, v in st.items()})
ok &= all(v["status"] in ("OK", "SKIPPED") for v in st.values())
sens = json.load(open("/kaggle/working/w3_export_evidence/offset_sensitivity_summary.json"))["groups"]
for k, g in sens.items():
    print(f"{k:28s} videos={g['videos']:4d} frac_iou>=0.5={g['frac_iou_ge_min']}  passes={g['passes']}")
for k in ("V|Drone|plus1", "V|Bird|plus1"):
    if not sens.get(k, {}).get("passes"):
        print("DUNG: nhom", k, "khong dat nguong -> KHONG chay run 03b/04, gui ket qua cho Claude"); ok = False
print("PASS" if ok else "FAIL — gui stage_status.json cho Claude, khong chay lai nhieu lan")

In [ ]:
for f in ("w3_review_v2.zip", "w3_export_evidence.zip"):
    p = f"/kaggle/working/{f}"; print(f, os.path.getsize(p) // 1024, "KB  sha256", sha(p))